# Train Ibiko to speak with your voice

You recorded yourself in Ibiko (**Settings › Chat › My voice**) and shared the recordings as `my-voice-recordings.zip`.
This notebook trains a fast **Piper** voice from them and gives you `ibiko-my-voice.zip` to add back in Ibiko.

**Before you start:** *Runtime › Change runtime type › T4 GPU* (free). Then run the cells in order (▶ on each, or *Runtime › Run all*).

It **fine-tunes** an existing English voice on your recordings, so 30–60 minutes of speech is enough, and training takes
about 1–3 hours on the free GPU. Keep the tab open while it trains.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU: choose Runtime > Change runtime type > T4 GPU, then run this again."

## 2. Install the Piper trainer

In [ ]:
!apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace -q
%cd /content

## 3. Upload your recordings
Choose the `my-voice-recordings.zip` you shared from Ibiko.

In [ ]:
import os, zipfile, glob
from google.colab import files
up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content/data')
root = os.path.dirname(glob.glob('/content/data/**/metadata.csv', recursive=True)[0])
wavs = os.path.join(root, 'wavs')
# the trainer's form: "file.wav|sentence" a line
rows = []
for line in open(os.path.join(root, 'metadata.csv'), encoding='utf-8'):
    if '|' in line:
        i, text = line.rstrip('\n').split('|', 1)
        if os.path.exists(os.path.join(wavs, i + '.wav')):
            rows.append(f'{i}.wav|{text}')
open('/content/train.csv', 'w', encoding='utf-8').write('\n'.join(rows) + '\n')
seconds = sum(os.path.getsize(os.path.join(wavs, r.split('|')[0])) - 44 for r in rows) / 2 / 22050
print(f'{len(rows)} recordings, {seconds/60:.1f} minutes of speech')
if seconds < 20 * 60:
    print('That is little: the voice will be rough. 45 minutes or so is better.')

## 4. Fetch an English voice to start from
Your voice is learned on top of it (the *lessac medium* voice), which is far quicker than starting from nothing.

In [ ]:
import re
from huggingface_hub import list_repo_files, hf_hub_download
repo = 'rhasspy/piper-checkpoints'
ckpts = [f for f in list_repo_files(repo, repo_type='dataset') if f.startswith('en/en_US/lessac/medium/') and f.endswith('.ckpt')]
print(ckpts)
base = hf_hub_download(repo, ckpts[0], repo_type='dataset', local_dir='/content/base')
base_epoch = int(re.search(r'epoch=(\d+)', ckpts[0]).group(1))
print('starting from', base, 'at epoch', base_epoch)

## 5. Train
`EXTRA_EPOCHS` is how long it learns your voice: 300–600 is usual. More sounds more like you, and takes longer.

In [ ]:
EXTRA_EPOCHS = 400
!cd /content/piper1-gpl && python3 -m piper.train fit \
  --data.voice_name my_voice \
  --data.csv_path /content/train.csv \
  --data.audio_dir {wavs} \
  --model.sample_rate 22050 \
  --data.espeak_voice en-us \
  --data.cache_dir /content/cache \
  --data.config_path /content/my_voice.onnx.json \
  --data.batch_size 16 \
  --ckpt_path {base} \
  --trainer.max_epochs {base_epoch + EXTRA_EPOCHS}

## 6. Make the voice file, and listen

In [ ]:
import glob, os
last = max(glob.glob('/content/piper1-gpl/lightning_logs/**/checkpoints/*.ckpt', recursive=True), key=os.path.getmtime)
print('trained:', last)
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint {last} --output-file /content/my_voice.onnx
!echo "Hello! I'm Ibiko, and this is my new voice. How does it sound?" | python3 -m piper -m /content/my_voice.onnx -f /content/sample.wav
from IPython.display import Audio
Audio('/content/sample.wav')

## 7. Prepare it for Ibiko, and download
Ibiko's voice engine (sherpa-onnx) reads the model with its details written inside and a `tokens.txt`.

In [ ]:
import json, onnx, zipfile
from google.colab import files
config = json.load(open('/content/my_voice.onnx.json', encoding='utf-8'))
with open('/content/tokens.txt', 'w', encoding='utf-8') as f:
    for symbol, ids in config['phoneme_id_map'].items():
        f.write(f'{symbol} {ids[0]}\n')
model = onnx.load('/content/my_voice.onnx')
meta = {
    'model_type': 'vits', 'comment': 'piper', 'language': 'English', 'voice': config['espeak']['voice'],
    'has_espeak': 1, 'n_speakers': config.get('num_speakers', 1), 'sample_rate': config['audio']['sample_rate'],
}
for k, v in meta.items():
    p = model.metadata_props.add(); p.key = k; p.value = str(v)
onnx.save(model, '/content/model.onnx')
with zipfile.ZipFile('/content/ibiko-my-voice.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/content/model.onnx', 'model.onnx')
    z.write('/content/tokens.txt', 'tokens.txt')
    z.write('/content/my_voice.onnx.json', 'config.json')
files.download('/content/ibiko-my-voice.zip')
print('Done: in Ibiko, Settings > Chat > My voice > Add my trained voice, and choose ibiko-my-voice.zip.')